<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Desafio 4, LSTM Bot QA

### Datos
El objetivo es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [1]:
# Instalar gdown si no está disponible
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", "gdown", "--quiet"])

0

In [2]:
import re
import json
import os

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences, to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout
)
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
# Descarga del dataset ConvAI2
import gdown

if not os.path.exists('data_volunteers.json'):
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download'
    gdown.download(url, 'data_volunteers.json', quiet=False)
else:
    print("El dataset ya se encuentra descargado")

Downloading...
From: https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN
To: H:\Mi unidad\CEIA\PLN - Procesamiento lenguaje natural\Desafios\Desafio4\data_volunteers.json
100%|██████████| 2.58M/2.58M [00:04<00:00, 637kB/s]


In [4]:
# Carga del dataset
with open('data_volunteers.json') as f:
    data = json.load(f)

print(f"Cantidad de diálogos: {len(data)}")
print(f"Campos disponibles: {data[0].keys()}")

Cantidad de diálogos: 1111
Campos disponibles: dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])


In [5]:
# Extracción de pares pregunta-respuesta
input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_len = 30

def clean_text(txt):
    txt = txt.lower()
    txt = txt.replace("'d", " had")
    txt = txt.replace("'s", " is")
    txt = txt.replace("'m", " am")
    txt = txt.replace("don't", "do not")
    txt = re.sub(r'\W+', ' ', txt).strip()
    return txt

for line in data:
    for i in range(len(line['dialog']) - 1):
        chat_in  = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i + 1]['text'])

        if len(chat_in.split()) >= max_len or len(chat_out.split()) >= max_len:
            continue

        input_sentences.append(chat_in)
        output_sentences.append(chat_out + ' <eos>')
        output_sentences_inputs.append('<sos> ' + chat_out)

print("Pares utilizados:", len(input_sentences))
print("Ejemplo entrada:", input_sentences[1])
print("Ejemplo salida: ", output_sentences[1])
print("Ejemplo dec_in: ", output_sentences_inputs[1])

Pares utilizados: 13323
Ejemplo entrada: hi how are you
Ejemplo salida:  not bad and you <eos>
Ejemplo dec_in:  <sos> not bad and you


### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

In [6]:
# Tokenización

# Tokenizador del encoder (entradas)
tokenizer_input = Tokenizer(filters='')
tokenizer_input.fit_on_texts(input_sentences)
word2idx_inputs = tokenizer_input.word_index
print(f"Vocabulario encoder: {len(word2idx_inputs)} palabras")

# Tokenizador del decoder (salidas + tokens especiales)
tokenizer_output = Tokenizer(filters='')
tokenizer_output.fit_on_texts(output_sentences + output_sentences_inputs)
word2idx_outputs = tokenizer_output.word_index
num_words_output = len(word2idx_outputs) + 1  # +1 por índice 0 reservado
print(f"Vocabulario decoder: {len(word2idx_outputs)} palabras")

#Secuencias y padding

encoder_input_sequences = tokenizer_input.texts_to_sequences(input_sentences)
max_input_len = max(len(s) for s in encoder_input_sequences)
encoder_input_sequences = pad_sequences(
    encoder_input_sequences, maxlen=max_input_len, padding='pre'
)

decoder_input_sequences = tokenizer_output.texts_to_sequences(output_sentences_inputs)
decoder_output_sequences_raw = tokenizer_output.texts_to_sequences(output_sentences)

max_out_len = max(
    max(len(s) for s in decoder_input_sequences),
    max(len(s) for s in decoder_output_sequences_raw)
)

decoder_input_sequences = pad_sequences(
    decoder_input_sequences, maxlen=max_out_len, padding='post'
)
decoder_output_sequences_raw = pad_sequences(
    decoder_output_sequences_raw, maxlen=max_out_len, padding='post'
)

# decoder_targets: one-hot para calcular la loss
decoder_targets = to_categorical(
    decoder_output_sequences_raw, num_classes=num_words_output
)

print(f"encoder_input_sequences : {encoder_input_sequences.shape}")
print(f"decoder_input_sequences : {decoder_input_sequences.shape}")
print(f"decoder_targets         : {decoder_targets.shape}")
print(f"max_input_len={max_input_len}  max_out_len={max_out_len}")

Vocabulario encoder: 3872 palabras
Vocabulario decoder: 3854 palabras
encoder_input_sequences : (13323, 29)
decoder_input_sequences : (13323, 30)
decoder_targets         : (13323, 30, 3855)
max_input_len=29  max_out_len=30


### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

In [8]:
import subprocess, sys, zipfile

EMBEDDING_DIM = 100
GLOVE_PATH = 'glove.6B.100d.txt'

# Descarga automática si no existe
if not os.path.exists(GLOVE_PATH):
    print("Descargando GloVe...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "requests", "--quiet"
    ])
    import requests
    url = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"
    zip_path = "glove.6B.zip"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extract(GLOVE_PATH)
    os.remove(zip_path)
    print("Descarga completa.")
else:
    print("GloVe ya disponible.")

# Cargar vectores GloVe
glove_embeddings = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

print(f"Vectores GloVe cargados: {len(glove_embeddings)}")

# Matriz de embedding para el encoder
num_words_input = len(word2idx_inputs) + 1
embedding_matrix_input = np.zeros((num_words_input, EMBEDDING_DIM))
for word, idx in word2idx_inputs.items():
    vec = glove_embeddings.get(word)
    if vec is not None:
        embedding_matrix_input[idx] = vec

# Matriz de embedding para el decoder
embedding_matrix_output = np.zeros((num_words_output, EMBEDDING_DIM))
for word, idx in word2idx_outputs.items():
    vec = glove_embeddings.get(word)
    if vec is not None:
        embedding_matrix_output[idx] = vec

print(f"Matriz encoder: {embedding_matrix_input.shape}")
print(f"Matriz decoder: {embedding_matrix_output.shape}")

Descargando GloVe...
Descarga completa.
Vectores GloVe cargados: 400000
Matriz encoder: (3873, 100)
Matriz decoder: (3855, 100)


### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder utilizando los datos generados en los puntos anteriores. Utilce como referencias los ejemplos vistos en clase.

In [9]:
LATENT_DIM = 256
BATCH_SIZE = 64
EPOCHS     = 20

# ENCODER
encoder_inputs = Input(shape=(max_input_len,), name='encoder_inputs')

enc_embedding = Embedding(
    input_dim=num_words_input,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix_input],
    trainable=False,
    name='enc_embedding'
)(encoder_inputs)

# return_state=True para obtener h y c y pasarlos al decoder
encoder_outputs, state_h, state_c = LSTM(
    LATENT_DIM, return_state=True, name='encoder_lstm'
)(enc_embedding)

encoder_states = [state_h, state_c]

# DECODER
decoder_inputs = Input(shape=(max_out_len,), name='decoder_inputs')

dec_embedding = Embedding(
    input_dim=num_words_output,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix_output],
    trainable=False,
    name='dec_embedding'
)(decoder_inputs)

# return_sequences=True para predecir token a token
decoder_lstm = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name='decoder_lstm'
)
decoder_outputs, _, _ = decoder_lstm(
    dec_embedding, initial_state=encoder_states
)

decoder_dense = Dense(num_words_output, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)

# ENTRENAMIENTO
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 29)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_embedding       │ (None, 29, 100)   │    387,300 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_embedding       │ (None, 30, 100)   │    385,500 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    365,568 │ enc_embedding[0]… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 30, 256), │    365,568 │ dec_embedding[0]… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 30, 3855)  │    990,735 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,494,671 (9.52 MB)

 Trainable params: 1,721,871 (6.57 MB)

 Non-trainable params: 772,800 (2.95 MB)

In [10]:
# Entrenamiento
history = model.fit(
    [encoder_input_sequences, decoder_input_sequences],
    decoder_targets,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1
)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 54s 271ms/step - accuracy: 0.7830 - loss: 1.4730 - val_accuracy: 0.8106 - val_loss: 1.1350
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 51s 271ms/step - accuracy: 0.8178 - loss: 1.0758 - val_accuracy: 0.8267 - val_loss: 1.0517
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 48s 255ms/step - accuracy: 0.8324 - loss: 0.9988 - val_accuracy: 0.8338 - val_loss: 0.9993
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 49s 262ms/step - accuracy: 0.8378 - loss: 0.9504 - val_accuracy: 0.8393 - val_loss: 0.9681
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 45s 239ms/step - accuracy: 0.8422 - loss: 0.9165 - val_accuracy: 0.8432 - val_loss: 0.9367
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 58s 309ms/step - accuracy: 0.8453 - loss: 0.8900 - val_accuracy: 0.8411 - val_loss: 0.9241
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 60s 318ms/step - accuracy: 0.8476 - loss: 0.8688 - val_accuracy: 0.8453 - val_loss: 0.9023
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 53s 280ms/step - accuracy: 0.8495 - loss: 0

### 5 - Inferencia
Experimentar el funcionamiento de su modelo. Recuerde que debe realizar la inferencia de los modelos por separado de encoder y decoder.

In [11]:
#MODELO DE INFERENCIA

# Encoder: recibe la entrada y devuelve los estados
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder: recibe token actual + estados anteriores
decoder_state_input_h = Input(shape=(LATENT_DIM,), name='dec_state_h')
decoder_state_input_c = Input(shape=(LATENT_DIM,), name='dec_state_c')
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reutiliza la capa de embedding del decoder
dec_emb_inf = dec_embedding  # misma capa entrenada

dec_out_inf, state_h_inf, state_c_inf = decoder_lstm(
    # Embedding de un solo token (paso a paso)
    Embedding(
        input_dim=num_words_output,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix_output],
        trainable=False
    )(Input(shape=(1,))),
    initial_state=decoder_states_inputs
)

#Reconstrucción correcta del decoder de inferencia
dec_inf_input = Input(shape=(1,), name='dec_inf_input')
dec_inf_emb   = model.get_layer('dec_embedding')(dec_inf_input)
dec_inf_out, inf_h, inf_c = decoder_lstm(
    dec_inf_emb, initial_state=decoder_states_inputs
)
dec_inf_out = decoder_dense(dec_inf_out)

decoder_model = Model(
    [dec_inf_input] + decoder_states_inputs,
    [dec_inf_out, inf_h, inf_c]
)

#Diccionarios inversos para decodificar índices → palabras
idx2word_input  = {v: k for k, v in word2idx_inputs.items()}
idx2word_output = {v: k for k, v in word2idx_outputs.items()}

print("Modelos de inferencia construidos.")

Modelos de inferencia construidos.


In [17]:
def decode_sequence(input_seq):
    """Genera una respuesta token a token dado un input_seq codificado."""
    # 1. Codificar la entrada y obtener estados del encoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # 2. Iniciar el decoder con el token <sos>
    sos_idx = word2idx_outputs.get('<sos>', 1)
    target_seq = np.array([[sos_idx]])

    stop_condition = False
    decoded_sentence = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Tomar el índice de mayor probabilidad
        sampled_idx  = np.argmax(output_tokens[0, -1, :])
        sampled_word = idx2word_output.get(sampled_idx, '')

        if sampled_word == '<eos>' or len(decoded_sentence) > max_out_len:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)

        # Actualizar secuencia y estados para el siguiente paso
        target_seq    = np.array([[sampled_idx]])
        states_value  = [h, c]

    return ' '.join(decoded_sentence)




In [20]:
def chat(text):
    # Limpiar y tokenizar igual que en el preprocesamiento
    text = clean_text(text)
    seq = tokenizer_input.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_input_len, padding='pre')
    response = decode_sequence(seq)
    return response



In [23]:
q1 = "hello how are you"
q2 = "what do you do for a living"
q3 = "do you have any pets"
q4 = "what kind of music do you like"
q5 = "do you have any brothers or sisters"
q6 = "what did you do last weekend"

for i, q in enumerate([q1, q2, q3, q4, q5, q6], 1):
    print(f"Q{i}: {q}")
    print(f"A{i}: {chat(q)}")
    print()

Q1: hello how are you
A1: i am fine

Q2: what do you do for a living
A2: i am a teacher

Q3: do you have any pets
A3: i do not have any pets

Q4: what kind of music do you like
A4: i like to go to the beach

Q5: do you have any brothers or sisters
A5: i am not sure what you mean

Q6: what did you do last weekend
A6: i am not sure what you mean

